In [1]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

from transformers import ViTForImageClassification

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


In [3]:
TEST_DIR = r"D:\new01\herb-recognition\dataset\test"


In [4]:
assert os.path.exists(TEST_DIR)

print("Folders inside test:")
for f in os.listdir(TEST_DIR):
    print(" -", f)


Folders inside test:
 - Adhatoda_vasica
 - Aegle_marmelos
 - Ailanthus_excelsa
 - Allium_cepa
 - Allium_sativum
 - Aloe_vera
 - Andrographis_paniculata
 - Anethum_graveolens
 - Asparagus_racemosus
 - Azadirachta_indica
 - Bacopa_monnieri
 - Berberis_aristata
 - Boerhavia_diffusa
 - Boswellia_serrata
 - Brassica_juncea
 - Carum_carvi
 - Cassia_fistula
 - Celastrus_paniculatus
 - Centella_asiatica
 - Cinnamomum_verum
 - Clitoria_ternatea
 - Commiphora_mukul
 - Convolvulus_pluricaulis
 - Cordia_dichotoma
 - Coriandrum_sativum
 - Cuminum_cyminum
 - Curcuma_longa
 - Desmodium_gangeticum
 - Elettaria_cardamomum
 - Emblica_officinalis
 - Ferula_asafoetida
 - Foeniculum_vulgare
 - Fumaria_indica
 - Glycyrrhiza_glabra
 - Gymnema_sylvestre
 - Hedychium_spicatum
 - Hemidesmus_indicus
 - Holarrhena_antidysenterica
 - Inula_racemosa
 - Lawsonia_inermis
 - Mentha_spicata
 - Moringa_oleifera
 - Myristica_fragrans
 - Nardostachys_jatamansi
 - Nigella_sativa
 - Ocimum_tenuiflorum
 - Operculina_turpethu

In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])


In [6]:
def is_image_file(path):
    return path.lower().endswith(('.jpg', '.jpeg', '.png'))

test_dataset = ImageFolder(
    root=TEST_DIR,
    transform=transform,
    is_valid_file=is_image_file
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

class_names = test_dataset.classes
num_classes = len(class_names)

print("Classes detected:", class_names)
print("Number of classes:", num_classes)


Classes detected: ['Adhatoda_vasica', 'Aegle_marmelos', 'Ailanthus_excelsa', 'Allium_cepa', 'Allium_sativum', 'Aloe_vera', 'Andrographis_paniculata', 'Anethum_graveolens', 'Asparagus_racemosus', 'Azadirachta_indica', 'Bacopa_monnieri', 'Berberis_aristata', 'Boerhavia_diffusa', 'Boswellia_serrata', 'Brassica_juncea', 'Carum_carvi', 'Cassia_fistula', 'Celastrus_paniculatus', 'Centella_asiatica', 'Cinnamomum_verum', 'Clitoria_ternatea', 'Commiphora_mukul', 'Convolvulus_pluricaulis', 'Cordia_dichotoma', 'Coriandrum_sativum', 'Cuminum_cyminum', 'Curcuma_longa', 'Desmodium_gangeticum', 'Elettaria_cardamomum', 'Emblica_officinalis', 'Ferula_asafoetida', 'Foeniculum_vulgare', 'Fumaria_indica', 'Glycyrrhiza_glabra', 'Gymnema_sylvestre', 'Hedychium_spicatum', 'Hemidesmus_indicus', 'Holarrhena_antidysenterica', 'Inula_racemosa', 'Lawsonia_inermis', 'Mentha_spicata', 'Moringa_oleifera', 'Myristica_fragrans', 'Nardostachys_jatamansi', 'Nigella_sativa', 'Ocimum_tenuiflorum', 'Operculina_turpethum', 

In [7]:
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=100
)

checkpoint = torch.load("model/vit_herb_model.pth", map_location=device)
model.load_state_dict(checkpoint)
model.to(device)
model.eval()

RuntimeError: Error(s) in loading state_dict for Linear:
	size mismatch for bias: copying a param with shape torch.Size([1000]) from checkpoint, the shape in current model is torch.Size([100]).

In [ ]:
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs.logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())


In [ ]:
accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")


In [ ]:
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"Precision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"F1-score: {f1 * 100:.2f}%")

In [ ]:
labels = list(range(100))

report = classification_report(
    y_true,
    y_pred,
    labels=labels,
    target_names=class_names,
    zero_division=0
)

print(report)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=labels)

plt.figure(figsize=(20, 20))
sns.heatmap(cm, xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - 100 Herb Classification")
plt.show()

In [ ]:
# plt.figure(figsize=(20, 20))

# sns.heatmap(
#     cm,
#     cmap="Blues",
#     square=True,
#     cbar=True,
#     xticklabels=class_names,
#     yticklabels=class_names
# )

# plt.xticks(rotation=90, fontsize=6)
# plt.yticks(rotation=0, fontsize=6)

# plt.xlabel("Predicted Label")
# plt.ylabel("True Label")
# plt.title("Confusion Matrix – ViT Medicinal Herb Classification")

# plt.tight_layout()
# plt.show()

plt.figure(figsize=(18, 18))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – ViT Medicinal Herb Classification")
plt.show()


In [ ]:
print("FINAL EVALUATION METRICS")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-score  : {f1 * 100:.2f}%")

In [8]:
import torch
state_dict = torch.load("model/vit_herb_model.pth", map_location="cpu")
print(state_dict["classifier.weight"].shape)

torch.Size([100, 768])
